In [11]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mj_sim/quadruped')

import numpy as np
import mujoco
import mujoco.viewer
import pinocchio as pin
from scipy.spatial import ConvexHull
from core    import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel, LEG_KEYS, LEG_IDX
from control import GaitScheduler, LegController, SingleRigidBodyDynamics, OSQP_Solvers
from utils   import SimScheduler, Visualizer

%load_ext autoreload
%autoreload 2

URDF = '/home/frlab/mj_opt/models/go2/urdf/go2_description.urdf'
PKG  = '/home/frlab/mj_opt/models/go2'
MJCF = '/home/frlab/mj_opt/models/scenes/go2/scene.xml'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
def compute_reference_trajectory(x_current, v_body_cmd, yaw_rate_cmd, z_target, N, dt):
    """
    x_current: 현재 상태 [Theta, p, omega, v, -1] (13,)
    v_body_cmd: 조종기 선속도 명령 [vx, vy, 0] (3,) - Body frame
    yaw_rate_cmd: 조종기 각속도 명령 (스칼라) - Body frame
    z_target: 로봇 몸통의 목표 높이 (스칼라)
    N: 예측 호라이즌 길이
    dt: MPC 제어 주기
    """
    X_ref = np.zeros(13 * N)
    
    # 현재 상태 추출
    roll_curr, pitch_curr, yaw_curr = x_current[0:3]
    px_curr, py_curr, pz_curr       = x_current[3:6]
    
    # 1. Body -> World 속도 변환 (현재 Yaw 각도 기준)
    cy, sy = np.cos(yaw_curr), np.sin(yaw_curr)
    Rz = np.array([
        [cy, -sy, 0],
        [sy,  cy, 0],
        [ 0,   0, 1]
    ])
    v_world_cmd = Rz @ v_body_cmd  # [vx_world, vy_world, 0]
    
    # 2. 호라이즌 N 스텝 동안의 미래 상태 생성
    for i in range(N):
        # i 스텝 뒤의 시간
        t_future = (i + 1) * dt
        
        # --- 참조 상태 계산 ---
        # 자세 (Roll, Pitch는 0 유지, Yaw는 현재 각도에서 목표 각속도만큼 회전)
        theta_ref = np.array([0.0, 0.0, yaw_curr + yaw_rate_cmd * t_future])
        
        # 위치 (XY는 현재 위치에서 목표 속도만큼 이동, Z는 목표 높이 유지)
        p_ref = np.array([
            px_curr + v_world_cmd[0] * t_future,
            py_curr + v_world_cmd[1] * t_future,
            z_target
        ])
        
        # 각속도 (Roll, Pitch 방향 각속도는 0, Yaw 방향만 존재)
        omega_ref = np.array([0.0, 0.0, yaw_rate_cmd])
        
        # 선속도 (World 좌표계 목표 속도)
        v_ref = np.array([v_world_cmd[0], v_world_cmd[1], 0.0])
        
        # 중력 상태 (-1)
        # (주의: 오차를 구할 때 x - x_ref 를 계산하므로, x_ref의 13번째를 0으로 둬야 
        # -1 - 0 = -1 이 되어 중력 트릭이 깨지지 않습니다. 혹은 가중치 Q의 13번째를 0으로 둡니다.)
        gravity_ref = 0.0
        
        # --- X_ref 벡터에 이어 붙이기 ---
        idx = i * 13
        X_ref[idx : idx+3]   = theta_ref
        X_ref[idx+3 : idx+6] = p_ref
        X_ref[idx+6 : idx+9] = omega_ref
        X_ref[idx+9 : idx+12]= v_ref
        X_ref[idx+12]        = gravity_ref
        
    return X_ref

In [13]:
state    = FloatingBaseRobotState()
wrapper  = Pinocchio_Wrapper(URDF, PKG)
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel   = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)
kernel.register_foot_geoms({"FL": "FL", "FR": "FR", "RL": "RL", "RR": "RR"})
gait     = GaitScheduler(gait_hz=1.0, duty=0.8)
leg_ctrl = LegController(wrapper, gait, kernel)

N_MPC  = 20
DT_MPC = 0.01
dynamics = SingleRigidBodyDynamics(wrapper, legs=LEG_KEYS, MPC_dt=DT_MPC, N=N_MPC)
# constraint: friction cone (4-face) + fz_min per leg per step  → 5*4*N rows
solver   = OSQP_Solvers(n=12*N_MPC, m=5*4*N_MPC)

# 속도 커맨드 (body frame)
VX_CMD       = 0.3   # m/s  전진
YAW_RATE_CMD = 0.0   # rad/s
Z_TARGET     = 0.27  # m  서있는 높이

# MPC weight
Q_MPC = np.diag([100, 100, 100,   # roll/pitch/yaw
                  10,  10,  10,   # px/py/pz
                   1,   1,   1,   # omega
                   1,   1,   1,   # vx/vy/vz
                   0])            # gravity trick → 가중치 0
R_MPC = 1e-4 * np.eye(12)

MU      = 0.7    # 마찰계수
FZ_MAX  = 660.0  # N

print("시스템 초기화 완료")
print(f"로봇 총 질량: {wrapper.m:.2f} kg")

시스템 초기화 완료
로봇 총 질량: 16.09 kg


Main Loop (상태 갱신, 제어, 뷰어, 시각화)

In [14]:
_setup_done = False

def _build_constraint(mask, N, mu, fz_max):
    """friction cone (4-face linearized) + fz bound, contact mask 반영."""
    # 단일 다리, 단일 스텝 A_leg (5, 3):
    #   friction cone: |fx| <= mu*fz, |fy| <= mu*fz  →  4 rows
    #   fz bound     : 0 <= fz <= fz_max             →  1 row  (합쳐서 부등식으로)
    A_leg = np.array([
        [ 1,  0, -mu],
        [-1,  0, -mu],
        [ 0,  1, -mu],
        [ 0, -1, -mu],
        [ 0,  0,  1 ],
    ])  # (5,3)

    n_f  = 12 * N   # 결정변수 수
    n_c  = 5 * 4 * N
    A    = np.zeros((n_c, n_f))
    l    = np.full(n_c, -np.inf)
    u    = np.zeros(n_c)

    for k in range(N):
        mask_k = mask   # 고정 gait mask (매 step 동일하게 단순화)
        for i in range(4):
            row = (k * 4 + i) * 5
            col = k * 12 + i * 3
            if mask_k[i] == 1:   # stance
                A[row:row+5, col:col+3] = A_leg
                l[row:row+4] = -np.inf
                u[row:row+4] = 0.0
                l[row+4]     = 0.0
                u[row+4]     = fz_max
            else:                # swing: force = 0
                A[row:row+5, col:col+3] = np.eye(5, 3)
                l[row:row+5] = 0.0
                u[row:row+5] = 0.0
    return A, l, u


def on_control(t):
    global _setup_done

    kernel.update_robot_state(state)
    wrapper.update_model(state.q, state.dq)

    # ── 현재 상태 벡터 x (13,) ──
    rpy     = state.compute_rpy_world()             # (3,) world
    p_com   = wrapper.p_com_W                       # (3,) world
    omega_W = wrapper.R_WB @ state.base_ang_vel     # body→world
    v_com   = wrapper.v_com_W                       # (3,) world
    x_now   = np.concatenate([rpy, p_com, omega_W, v_com, [-1.0]])  # (13,)

    # ── 참조 궤적 ──
    v_body_cmd = np.array([VX_CMD, 0.0, 0.0])
    x_des = compute_reference_trajectory(x_now, v_body_cmd, YAW_RATE_CMD, Z_TARGET, N_MPC, DT_MPC)

    # ── SRBD 시스템 행렬 갱신 ──
    dynamics.continuous_system()
    dynamics.discrete_system()
    for k in range(N_MPC):
        dynamics.A_arr[k] = dynamics.A_d
        dynamics.B_arr[k] = dynamics.B_d

    H, g_vec = dynamics.quadratic_form_system(x_now, x_des, Q_MPC, R_MPC)

    # ── Constraint 조립 + solve ──
    mask = gait.get_mask(t)
    A_c, l_c, u_c = _build_constraint(mask, N_MPC, MU, FZ_MAX)

    if not _setup_done:
        solver.setup(H, g_vec, A_c, l_c, u_c)
        _setup_done = True
    else:
        solver.update(H=H, q=g_vec, A=A_c, l=l_c, u=u_c)

    f_opt = solver.solve()   # (12*N,) — 첫 스텝만 사용

    # ── MPC → joint torque ──
    if f_opt is not None:
        f0 = f_opt[:12]   # 현재 스텝 GRF (12,)
        tau_mpc = np.zeros(12)
        for leg in LEG_KEYS:
            i   = LEG_IDX[leg]
            J   = wrapper.compute_J_W(leg)
            J3  = J[:3, 6:][:, i*3 : i*3+3]   # (3,3)
            tau_mpc[i*3 : i*3+3] = J3.T @ f_opt[i*3 : i*3+3]
        kernel.ctrl_tau = tau_mpc
    else:
        # fallback: PD
        kernel.ctrl_tau = leg_ctrl.compute(t)

def on_render(self, t):
    pass

In [15]:
kernel.reset_to_keyframe()
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=on_control, on_render=on_render)

[SimScheduler] sim lagging wall-clock by 58.7 ms
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for partial updates of A
ERROR in osqp_update_data_mat: index vector is required for 